# 08 — Near-optimal and final-task registry

**Objective.** Freeze label-specific deployment members, matched F36-reference configurations, bootstrap-refit tasks, task chunks, and released seed/background/audit registries.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

The term “empirical near-optimal set” is used; the sampled search is not silently called a formal Rashomon set.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("08", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
import os
import pandas as pd
from cruxvc.io import read_table, write_json, write_table
from cruxvc.workflow import assign_task_chunks, build_refit_registry

near = read_table(P.models / "near_optimal_registry.parquet")
calibrated = read_table(P.models / "calibration_registry.parquet")
seeds = pd.read_csv(P.protocol / "seed_registry.csv")
CTX.recorder.inputs.extend([P.models / "near_optimal_registry.parquet", P.models / "calibration_registry.parquet", P.protocol / "seed_registry.csv"])

In [ ]:
f36_reference = (
    calibrated[(calibrated["analysis_role"].eq("matched_reference")) & calibrated["outcome"].eq("F36")]
    .sort_values(["platt_oof_log_loss", "family", "config_id", "seed"])
    .groupby("family", as_index=False).first()
)
reference_configs = f36_reference[["family", "config_id", "seed", "parameters_json", "platt_oof_log_loss"]].copy()
reference_path = write_table(reference_configs, P.models / "matched_reference_configs.parquet")

deployment = calibrated.copy()
deployment["task_id"] = "deployment__" + deployment["artifact_id"]
deployment["task_type"] = "deployment"
deployment["refit_id"] = "deployment"
deployment["analysis_role"] = deployment["analysis_role"].replace({
    "matched_reference": "matched_reference_deployment",
    "label_specific_near_optimal": "label_specific_near_optimal_deployment",
})
deployment["parameters_json"] = deployment["parameters_json"].astype(str)
deployment["refit_slot"] = -1
deployment["calibrated_model_path"] = deployment["calibrated_model_path"].astype(str)
deployment_tasks = deployment[[
    "task_id", "outcome", "family", "config_id", "parameters_json", "refit_id", "refit_slot", "seed",
    "task_type", "analysis_role", "calibrated_model_path"
]]

In [ ]:
bootstrap_tasks = build_refit_registry(
    reference_configs,
    CFG["outcomes"]["confirmatory"],
    int(PROFILE["bootstrap_refits"]),
    seeds,
)
bootstrap_tasks["task_type"] = "bootstrap"
bootstrap_tasks["analysis_role"] = "matched_reference_bootstrap"
bootstrap_tasks["calibrated_model_path"] = pd.NA
all_tasks = pd.concat([deployment_tasks, bootstrap_tasks], ignore_index=True)
requested_chunks = int(os.environ.get("CRUX_N_CHUNKS", "20"))
n_chunks = min(max(1, requested_chunks), max(1, len(all_tasks)))
all_tasks = assign_task_chunks(all_tasks, n_chunks)
tasks_path = write_table(all_tasks, P.protocol / "final_task_registry.parquet")
chunk_plan = all_tasks.groupby(["chunk_index", "task_type"]).size().unstack(fill_value=0).reset_index()
chunk_path = write_table(chunk_plan, P.protocol / "final_task_chunk_plan.csv")

In [ ]:
f36_global_family_count = near[(near["outcome"].eq("F36")) & near["near_optimal_global"]]["family"].nunique()
registry_audit = {
    "profile": PROFILE["name"],
    "bootstrap_refits_per_reference_config": int(PROFILE["bootstrap_refits"]),
    "deployment_tasks": int((all_tasks["task_type"] == "deployment").sum()),
    "bootstrap_tasks": int((all_tasks["task_type"] == "bootstrap").sum()),
    "n_chunks": int(n_chunks),
    "f36_global_near_optimal_family_count": int(f36_global_family_count),
    "two_family_gate_pass": bool(f36_global_family_count >= 2),
}
audit_path = write_json(registry_audit, P.audits / "08_final_registry_audit.json")
CTX.recorder.complete([reference_path, tasks_path, chunk_path, audit_path])
print(registry_audit)
print(chunk_plan.to_string(index=False))